## 기본 예시: 프롬프트 + 모델 + 출력 파서

가장 기본적이고 일반적인 사용 사례는 prompt 템플릿과 모델을 함께 연결하는 것입니다. 이것이 어떻게 작동하는지 보기 위해, 각 나라별 수도를 물어보는 Chain을 생성해 보겠습니다.


In [1]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH01-Basic")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH01-Basic


## 프롬프트 템플릿의 활용

`PromptTemplate`

- 사용자의 입력 변수를 사용하여 완전한 프롬프트 문자열을 만드는 데 사용되는 템플릿입니다
- 사용법
  - `template`: 템플릿 문자열입니다. 이 문자열 내에서 중괄호 `{}`는 변수를 나타냅니다.
  - `input_variables`: 중괄호 안에 들어갈 변수의 이름을 리스트로 정의합니다.

`input_variables`

- input_variables는 PromptTemplate에서 사용되는 변수의 이름을 정의하는 리스트입니다.

In [3]:
from langchain_teddynote.messages import stream_response  # 스트리밍 출력
from langchain_core.prompts import PromptTemplate

`from_template()` 메소드를 사용하여 PromptTemplate 객체 생성


In [4]:
# template 정의
template = "{country}의 수도는 어디인가요?"

# from_template 메소드를 이용하여 PromptTemplate 객체 생성
prompt_template = PromptTemplate.from_template(template)
prompt_template

PromptTemplate(input_variables=['country'], template='{country}의 수도는 어디인가요?')

In [5]:
# prompt 생성
prompt = prompt_template.format(country="대한민국")
prompt

'대한민국의 수도는 어디인가요?'

In [6]:
# prompt 생성
prompt = prompt_template.format(country="미국")
prompt

'미국의 수도는 어디인가요?'

In [7]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-3.5-turbo",
    max_tokens=2048,
    temperature=0.1,
)

## Chain 생성

### LCEL(LangChain Expression Language)

![lcel.png](./images/lcel.png)

여기서 우리는 LCEL을 사용하여 다양한 구성 요소를 단일 체인으로 결합합니다

```
chain = prompt | model | output_parser
```

`|` 기호는 [unix 파이프 연산자](<https://en.wikipedia.org/wiki/Pipeline_(Unix)>)와 유사하며, 서로 다른 구성 요소를 연결하고 한 구성 요소의 출력을 다음 구성 요소의 입력으로 전달합니다.

이 체인에서 사용자 입력은 프롬프트 템플릿으로 전달되고, 그런 다음 프롬프트 템플릿 출력은 모델로 전달됩니다. 각 구성 요소를 개별적으로 살펴보면 무슨 일이 일어나고 있는지 이해할 수 있습니다.


In [20]:
# prompt 를 PromptTemplate 객체로 생성합니다.
# prompt = PromptTemplate.from_template("{topic} 에 대해 {how} 설명해주세요.")
prompt = PromptTemplate.from_template("{topic} 에 대해 간단하게 설명해주세요.")

model = ChatOpenAI()

chain = prompt | model

print(chain)

first=PromptTemplate(input_variables=['topic'], template='{topic} 에 대해 간단하게 설명해주세요.') last=ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x11fdc2790>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x11df5e8d0>, openai_api_key=SecretStr('**********'), openai_proxy='')


### invoke() 호출

- python 딕셔너리 형태로 입력값을 전달합니다.(키: 값)
- invoke() 함수 호출 시, 입력값을 전달합니다.

In [21]:
# input 딕셔너리에 주제를 '인공지능 모델의 학습 원리'으로 설정합니다.
input = {"topic": "인공지능 모델의 학습 원리"}
# input = {"topic": "인공지능 모델의 학습 원리", "how": "간단하게"}

In [17]:
# prompt 객체와 model 객체를 파이프(|) 연산자로 연결하고 invoke 메서드를 사용하여 input을 전달합니다.
# 이를 통해 AI 모델이 생성한 메시지를 반환합니다.
chain.invoke(input)
# chain.invoke("인공지능 모델의 학습 원리") # * 변수가 하나일때는 이렇게 넣어도 된다.

AIMessage(content='인공지능 모델의 학습은 주어진 데이터를 이용하여 모델의 파라미터를 조정하는 과정입니다. 이를 위해서는 먼저 모델의 구조를 정의하고, 데이터를 입력으로 사용하여 모델이 정확한 출력을 내도록 파라미터를 조정해야 합니다. \n\n일반적으로 학습은 손실 함수를 최소화하는 방향으로 진행됩니다. 손실 함수는 모델의 출력과 실제 정답 사이의 차이를 계산하는 함수로, 이를 최소화하는 방향으로 모델의 파라미터를 업데이트하면서 학습이 진행됩니다. 이러한 파라미터 업데이트는 주로 경사 하강법을 이용하여 수행되며, 학습 데이터를 여러 번 반복하여 모델을 학습시킵니다. \n\n이렇게 학습된 모델은 새로운 데이터에 대해 정확한 예측을 할 수 있게 되며, 학습 데이터에 대한 정확도를 높이는 것이 주요 목표입니다.', response_metadata={'token_usage': {'completion_tokens': 315, 'prompt_tokens': 35, 'total_tokens': 350}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-6a260192-5d38-4737-9763-cdf59fd4d427-0', usage_metadata={'input_tokens': 35, 'output_tokens': 315, 'total_tokens': 350})

아래는 스트리밍을 출력하는 예시 입니다.

In [ ]:
# 스트리밍 출력을 위한 요청
answer = chain.stream(input)
# 스트리밍 출력
stream_response(answer)

### 출력파서(Output Parser)


In [22]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [28]:
import inspect
from langchain_core.output_parsers import StrOutputParser

print(inspect.getfile(StrOutputParser))

/Users/yglee/Library/Caches/pypoetry/virtualenvs/langchain-kr-U9Y9d8F_-py3.11/lib/python3.11/site-packages/langchain_core/output_parsers/string.py


Chain 에 출력파서를 추가합니다.

In [23]:
# 프롬프트, 모델, 출력 파서를 연결하여 처리 체인을 구성합니다.
chain = prompt | model | output_parser

In [24]:
# chain 객체의 invoke 메서드를 사용하여 input을 전달합니다.
input = {"topic": "인공지능 모델의 학습 원리"}
chain.invoke(input)

'인공지능 모델의 학습 원리는 데이터를 입력으로 받아 내부의 가중치를 조정하여 원하는 결과를 출력하는 과정입니다. 이 과정은 크게 입력층, 은닉층, 출력층으로 구성된 신경망을 활용하여 이루어집니다. \n\n먼저, 입력층에서 데이터가 모델에 입력되고, 이 데이터는 은닉층을 거쳐 출력층으로 전달됩니다. 은닉층은 입력층의 데이터를 받아 가중치를 곱하고 활성화 함수를 통과시켜 새로운 값을 생성합니다. 이러한 과정을 여러 번 반복하여 가중치를 조정하고 최적의 결과를 도출합니다.\n\n학습의 목표는 입력 데이터와 실제 결과값 간의 차이를 최소화하는 것이며, 이를 위해 손실 함수를 활용하여 모델의 성능을 평가하고 가중치를 업데이트합니다. 이러한 과정을 반복하여 모델이 학습하고 예측을 수행할 수 있도록 합니다.'

In [ ]:
# 스트리밍 출력을 위한 요청
answer = chain.stream(input)
# 스트리밍 출력
stream_response(answer)

### 템플릿을 변경하여 적용

- 아래의 프롬프트 내용을 얼마든지 **변경** 하여 테스트 해볼 수 있습니다.
- `model_name` 역시 변경하여 테스트가 가능합니다.

In [32]:
template = """
당신은 영어를 가르치는 10년차 영어 선생님입니다. 주어진 상황에 맞는 영어 회화를 작성해 주세요.
양식은 [FORMAT]을 참고하여 작성해 주세요.

#상황:
{question}

#FORMAT:
- 영어 회화:
- 한글 해석:
"""

# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다.
model = ChatOpenAI(model_name="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

In [33]:
# 체인을 구성합니다.
chain = prompt | model | output_parser

In [31]:
# 완성된 Chain을 실행하여 답변을 얻습니다.
print(chain.invoke({"question": "저는 식당에 가서 음식을 주문하고 싶어요"}))

- 영어 회화:  
  **Customer**: Hi, could I see the menu, please?  
  **Waiter**: Of course, here you go. Let me know if you have any questions about the menu.  
  **Customer**: Thank you. I'll need a few minutes to decide.  
  **Waiter**: Take your time. I'll be back in a few minutes to take your order.  
  **Customer**: Actually, I'm ready to order now. I'd like the grilled salmon and a side of mashed potatoes, please.  
  **Waiter**: Great choice! Would you like anything to drink?  
  **Customer**: Yes, I'll have a glass of white wine, please.  
  **Waiter**: Perfect, I'll get that order in for you right away.

- 한글 해석:
  **손님**: 안녕하세요, 메뉴판 좀 볼 수 있을까요?
  **웨이터**: 물론이죠, 여기 있습니다. 메뉴에 대해 궁금한 점이 있으시면 알려주세요.
  **손님**: 감사합니다. 조금 시간이 필요할 것 같아요.
  **웨이터**: 천천히 결정하세요. 몇 분 후에 주문 받으러 다시 올게요.
  **손님**: 사실 지금 주문할 준비가 되었어요. 그릴에 구운 연어와 마시 포테이토를 주문할게요.
  **웨이터**: 좋은 선택이네요! 음료는 뭘로 드릴까요?
  **손님**: 네, 화이트 와인 한 잔 주세요.
  **웨이터**: 알겠습니다, 바로 주문 넣어 드릴게요.


In [34]:
# 완성된 Chain을 실행하여 답변을 얻습니다.
# 스트리밍 출력을 위한 요청
answer = chain.stream({"question": "저는 식당에 가서 음식을 주문하고 싶어요"})
# 스트리밍 출력
stream_response(answer)

- 영어 회화:
  - Waiter: Good evening! Welcome to our restaurant. How many people are in your party?
  - You: Good evening! It's just me, one person.
  - Waiter: Great! Here’s the menu. Can I start you off with something to drink?
  - You: Yes, I’d like a glass of water, please.
  - Waiter: Sure! Are you ready to order your food?
  - You: Yes, I’d like the grilled chicken salad, please.
  - Waiter: Would you like any dressing with that?
  - You: Yes, please. I’ll have balsamic vinaigrette.
  - Waiter: Perfect! I’ll get that order in for you right away.

- 한글 해석:
  - 웨이터: 좋은 저녁입니다! 저희 식당에 오신 것을 환영합니다. 몇 분이신가요?
  - 당신: 좋은 저녁입니다! 저 혼자입니다.
  - 웨이터: 좋습니다! 여기 메뉴입니다. 음료는 무엇을 드릴까요?
  - 당신: 네, 물 한 잔 주세요.
  - 웨이터: 알겠습니다! 음식을 주문할 준비가 되셨나요?
  - 당신: 네, 그릴 치킨 샐러드로 주문할게요.
  - 웨이터: 드레싱은 어떤 걸 원하시나요?
  - 당신: 네, 발사믹 비네그레트를 주세요.
  - 웨이터: 완벽합니다! 즉시 주문하겠습니다.

In [35]:
# 이번에는 question 을 '미국에서 피자 주문'으로 설정하여 실행합니다.
# 스트리밍 출력을 위한 요청
answer = chain.stream({"question": "미국에서 피자 주문"})
# 스트리밍 출력
stream_response(answer)

- 영어 회화:
**Customer:** Hi, I’d like to order a pizza, please.  
**Pizza Shop:** Sure! What size would you like?  
**Customer:** I’ll have a large, please.  
**Pizza Shop:** Great! What toppings do you want?  
**Customer:** Could I get pepperoni, mushrooms, and extra cheese?  
**Pizza Shop:** Sounds good! Would you like anything to drink?  
**Customer:** Yes, a medium cola, please.  
**Pizza Shop:** Perfect! Your total comes to $25. Would you like to pay with cash or card?  
**Customer:** I’ll pay with a card.  
**Pizza Shop:** Alright, your order will be ready in about 30 minutes. Thank you!  
**Customer:** Thank you!

- 한글 해석:
**고객:** 안녕하세요, 피자를 주문하고 싶어요.  
**피자 가게:** 네, 어떤 사이즈로 드릴까요?  
**고객:** 큰 사이즈로 주세요.  
**피자 가게:** 좋습니다! 어떤 토핑을 원하시나요?  
**고객:** 페퍼로니, 버섯, 그리고 치즈 추가로 주세요.  
**피자 가게:** 좋네요! 음료수는 필요하신가요?  
**고객:** 네, 중간 사이즈 콜라 하나 주세요.  
**피자 가게:** 완벽합니다! 총 금액은 25달러입니다. 현금으로 결제하시겠습니까, 아니면 카드로 하시겠습니까?  
**고객:** 카드로 결제할게요.  
**피자 가게:** 알겠습니다. 주문은 약 30분 후에 준비될 거예요. 감사합니다!  
**고객:** 감사합니다!